[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanemat/uol-fp/blob/main/proto2/2pipeline.ipynb)
Confirmed runtime version: 2026.04

## Setup

In [ ]:
import sys

vi = sys.version_info
if not ((3, 12) <= (vi.major, vi.minor) < (3, 13)):
    raise RuntimeError(f"Python 3.12 required, got {vi.major}.{vi.minor}.{vi.micro}")

print(f"Python {vi.major}.{vi.minor}.{vi.micro} ✓")

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

print("Setup complete.")

## Data Models

In [ ]:
class Role(str, Enum):
    TECHNICAL_METHOD = "technical_method"
    TASK = "task"
    DATASET = "dataset"
    EVALUATION_METRIC = "evaluation_metric"
    OTHER = "other"


@dataclass
class CandidateWithContext:
    candidate: str
    sentence: str
    section: str = "unknown"


@dataclass
class MethodologyProfile:
    technical_method: list[str] = field(default_factory=list)
    task: list[str] = field(default_factory=list)
    dataset: list[str] = field(default_factory=list)
    evaluation_metric: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "TechnicalMethod": self.technical_method,
            "Task": self.task,
            "Dataset": self.dataset,
            "EvaluationMetric": self.evaluation_metric,
        }


print("Models ready.")

## Step 0 — Load TEI XML

Upload a TEI XML file produced by local GROBID.

In [ ]:
from xml.etree import ElementTree as ET

from google.colab import files

NS = {"tei": "http://www.tei-c.org/ns/1.0"}
SKIP_HEADINGS = {"references", "acknowledgements", "acknowledgments"}


def _text(element) -> str:
    return " ".join(element.itertext()).strip()


uploaded = files.upload()
xml_filename = next(iter(uploaded))
xml_bytes = uploaded[xml_filename]

root = ET.fromstring(xml_bytes.decode("utf-8"))

abstract_el = root.find(".//tei:abstract", NS)
abstract_text = _text(abstract_el) if abstract_el is not None else ""

sections = []
for div in root.findall(".//tei:body//tei:div", NS):
    heading = div.findtext("tei:head", namespaces=NS) or ""
    if heading.lower().strip() in SKIP_HEADINGS:
        continue
    body = " ".join(_text(p) for p in div.findall("tei:p", NS)).strip()
    if body:
        sections.append({"heading": heading, "text": body})

if abstract_text:
    sections.insert(0, {"heading": "Abstract", "text": abstract_text})

print(f"Loaded : {xml_filename}")
print(f"Sections: {len(sections)}")
for s in sections:
    print(f"  - {s['heading']}")

## Step 1 — NLI Model Setup

In [ ]:
!pip install -q transformers torch

from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
)
print("Model loaded.")

## Smoke Test

Classify one sentence against 4 methodology labels.

In [ ]:
sentence = "We fine-tuned BERT on the SQuAD dataset and measured F1 score."
labels = ["technical method", "dataset", "evaluation metric", "task"]

result = classifier(sentence, candidate_labels=labels)
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.3f}  {label}")